# CC1B screening — search-sample precision / recall / F1

LLM predictions from run `l0_cc1_search-sample_010726_medium_f1` scored against human truth
(`data/consistency-check-datasets/screening/search-sample/search-sample.csv`).
Predictions are derived with the exact screening pipeline in `evals_local`. ELIGIBLE is the positive class.

In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path("../..").resolve())) 
from evals_local.loaders import load_screening_predictions
from evals_local.run import run_task
from evals_local.normalizers import normalize_screening_label

## 1. Derive pred labels from the batch outputs (no truth yet)

`load_screening_predictions` merges the batch records with the LLM outputs on `UT`, reads the
per-stage results `s1_r..s4_r`, and sets `pred = NOT_ELIGIBLE` if any stage is 0 else `ELIGIBLE`.

In [41]:
RUN = "l0_cc1_search-sample_010726_medium_f1"
TRUTH_PATH = Path("../../data/consistency-check-datasets/screening/search-sample/search-sample.csv")

eval_summary = run_task("screening", RUN, label_path=TRUTH_PATH)
print(f"saved eval data: {eval_summary['data_path']}")
for label, path in eval_summary["metric_paths"].items():
    print(f"saved eval metrics ({label}): {path}")


preds = load_screening_predictions(RUN)
preds["custom_id"] = preds["UT"].astype(str).str.strip()
preds["pred_label"] = preds["pred"].map(normalize_screening_label)

print(f"prediction rows: {len(preds)}")
assert preds["custom_id"].is_unique, "duplicate UT in predictions"
preds["pred_label"].value_counts()

saved eval data: /Users/hunain/d/coding-projects/biodiversity/data/labels/eval/l0_cc1_search-sample_010726_medium_f1/data/screening.xlsx
saved eval metrics (eligibility): /Users/hunain/d/coding-projects/biodiversity/data/labels/eval/l0_cc1_search-sample_010726_medium_f1/metrics/screening_eligibility_label_metrics.xlsx
saved eval metrics (eligibility_confusion): /Users/hunain/d/coding-projects/biodiversity/data/labels/eval/l0_cc1_search-sample_010726_medium_f1/metrics/screening_eligibility_confusion.xlsx
prediction rows: 400


pred_label
NOT_ELIGIBLE    231
ELIGIBLE        169
Name: count, dtype: int64

## 2. Load human truth and join on WOS ID (grain guarded)

The truth `eligibility` column is normalized with the *same* normalizer as the prediction so both
sides use the identical `ELIGIBLE` / `NOT_ELIGIBLE` format. The join asserts every prediction finds
exactly one truth row (no missing matches, no grain mismatch).

In [42]:
truth = pd.read_csv(TRUTH_PATH)
truth["custom_id"] = truth["custom_id"].astype(str).str.strip()
truth["true_label"] = truth["eligibility"].map(normalize_screening_label)

assert truth["custom_id"].is_unique, "duplicate custom_id in truth (grain mismatch)"

df = preds.merge(
    truth[["custom_id", "true_label", "Article Title"]],
    on="custom_id",
    how="inner",
    validate="one_to_one",
)

unmatched = len(preds) - len(df)
assert unmatched == 0, f"{unmatched} predictions had no matching truth row"
print(f"joined rows: {len(df)} | empty truth: {int(df['true_label'].eq('').sum())}")
df[["pred_label", "true_label"]].value_counts().reset_index(name="count")

joined rows: 400 | empty truth: 0


,pred_label,true_label,count
0,NOT_ELIGIBLE,NOT_ELIGIBLE,219
1,ELIGIBLE,ELIGIBLE,109
2,ELIGIBLE,NOT_ELIGIBLE,60
3,NOT_ELIGIBLE,ELIGIBLE,12


## 3. Confusion matrix + precision / recall / F1 (ELIGIBLE = positive)

In [43]:
scored = df[df["true_label"] != ""].copy()

pred_pos = scored["pred_label"] == "ELIGIBLE"
true_pos = scored["true_label"] == "ELIGIBLE"

tp = int((pred_pos & true_pos).sum())
fp = int((pred_pos & ~true_pos).sum())
fn = int((~pred_pos & true_pos).sum())
tn = int((~pred_pos & ~true_pos).sum())

precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

confusion = pd.DataFrame(
    [[tp, fp], [fn, tn]],
    index=pd.Index(["pred ELIGIBLE", "pred NOT_ELIGIBLE"], name="prediction"),
    columns=pd.Index(["true ELIGIBLE", "true NOT_ELIGIBLE"], name="truth"),
)
confusion

truth,true ELIGIBLE,true NOT_ELIGIBLE
prediction,,
pred ELIGIBLE,109,60
pred NOT_ELIGIBLE,12,219


In [44]:
pd.DataFrame([
    {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "positive_support": tp + fn,
        "scored_rows": len(scored),
    }
])

,tp,fp,fn,tn,precision,recall,f1,positive_support,scored_rows
0,109,60,12,219,0.64497,0.900826,0.751724,121,400


## 4. Disagreements 

In [45]:
df.loc[df["pred_label"] != df["true_label"], ["custom_id", "Article Title", "pred_label", "true_label"]]

,custom_id,Article Title,pred_label,true_label
2,WOS:000837210900001,A lesson for planning rodent eradications: int...,NOT_ELIGIBLE,ELIGIBLE
9,WOS:000381529700024,Antioxidant responses in estuarine invertebrat...,NOT_ELIGIBLE,ELIGIBLE
13,WOS:000645256100020,Assembly of species' climatic niches of coasta...,NOT_ELIGIBLE,ELIGIBLE
25,WOS:000836331500007,Change in Atlantic cod migrations and adaptabi...,NOT_ELIGIBLE,ELIGIBLE
52,WOS:001149833200001,Fine-scale spatial variation of northern shrim...,NOT_ELIGIBLE,ELIGIBLE
...,...,...,...,...
379,WOS:001355592200001,Relationship of seeding rate to biological nit...,ELIGIBLE,NOT_ELIGIBLE
381,WOS:000596128900009,Remediation of vanadium-contaminated soils by ...,ELIGIBLE,NOT_ELIGIBLE
384,WOS:001582512000001,Reproductive vulnerability of Tenualosa Ilisha...,ELIGIBLE,NOT_ELIGIBLE
393,WOS:000367598200013,Seasonally varied controls of climate and phen...,ELIGIBLE,NOT_ELIGIBLE


## 5. False positive audit: `-1` stage inclusions

The production screening loader currently labels a record as `NOT_ELIGIBLE` only when at least one
stage result is exactly `0`; records with `-1` in one or more stages are still treated as eligible.
This section quantifies how many false positives come from that ambiguity path.


In [46]:
stage_cols = ["s1_r", "s2_r", "s3_r", "s4_r"]

false_positives = scored[
    (scored["pred_label"] == "ELIGIBLE")
    & (scored["true_label"] == "NOT_ELIGIBLE")
].copy()

false_positives["has_minus_one"] = false_positives[stage_cols].eq(-1).any(axis=1)
false_positives["minus_one_stages"] = false_positives.apply(
    lambda row: ", ".join(col for col in stage_cols if row[col] == -1) or "none",
    axis=1,
)

minus_one_summary = pd.DataFrame(
    [
        {
            "fp_total": len(false_positives),
            "fp_with_any_minus_one": int(false_positives["has_minus_one"].sum()),
            "fp_without_minus_one": int((~false_positives["has_minus_one"]).sum()),
        }
    ]
)
minus_one_summary


,fp_total,fp_with_any_minus_one,fp_without_minus_one
0,60,17,43


In [48]:
minus_one_by_stage = (
    false_positives[stage_cols]
    .eq(-1)
    .sum()
    .rename("fp_minus_one_count")
    .reset_index(name="stage")
)

minus_one_patterns = (
    false_positives["minus_one_stages"]
    .value_counts()
    .rename_axis("minus_one_pattern")
    .reset_index(name="fp_count")
)

stage_tuple_counts = (
    false_positives
    .groupby(stage_cols, dropna=False)
    .size()
    .sort_values(ascending=False)
    .rename("fp_count")
    .reset_index()
)

minus_one_by_stage, minus_one_patterns, stage_tuple_counts


(  index  stage
 0  s1_r      0
 1  s2_r     14
 2  s3_r      1
 3  s4_r      4,
   minus_one_pattern  fp_count
 0              none        43
 1              s2_r        12
 2        s2_r, s4_r         2
 3              s4_r         2
 4              s3_r         1,
    s1_r  s2_r  s3_r  s4_r  fp_count
 0     1     1     1     1        43
 1     1    -1     1     1        12
 2     1    -1     1    -1         2
 3     1     1     1    -1         2
 4     1     1    -1     1         1)

In [49]:
title_col = "Article Title" if "Article Title" in false_positives.columns else "title"
requested_cols = [
    "custom_id",
    title_col,
    "minus_one_stages",
    *stage_cols,
    "s1_bio",
    "s2_dir",
    "s3_drivers",
    "s4_link",
]
display_cols = [col for col in requested_cols if col in false_positives.columns]
sort_cols = [col for col in ["minus_one_stages", title_col] if col in false_positives.columns]

false_positives.loc[
    false_positives["has_minus_one"],
    display_cols,
].sort_values(sort_cols)


,custom_id,Article Title,minus_one_stages,s1_r,s2_r,s3_r,s4_r,s1_bio,s2_dir,s3_drivers,s4_link
131,WOS:000295992900011,A meta-analysis of the effects of clearcut and...,s2_r,1,-1,1,1,[ecosystem_change],unclear,"[clearcutting, variable-retention harvesting]",direct_primary_evidence
197,WOS:000760823900001,CRISPR/Cas9-Targeted Mutagenesis of BnaFAE1 Ge...,s2_r,1,-1,1,1,[genetic_change],unclear,[genetic modification (CRISPR/Cas9 gene editing)],direct_primary_evidence
176,WOS:000419952800010,Checklist of non-native benthic macroinvertebr...,s2_r,1,-1,1,1,"[species_change, community_change]",unclear,"[inland waterways transport corridors, invasiv...",plausibly_implied
203,WOS:000446289100045,Delineation of metals and radionuclides biocon...,s2_r,1,-1,1,1,[ecosystem_change],unclear,"[climate_change, pollution]",direct_primary_evidence
257,WOS:000812361200002,Functional imbalance not functional evenness i...,s2_r,1,-1,1,1,[community_change],unclear,[agriculture and aquaculture],direct_primary_evidence
270,WOS:001200767900001,Highly structured populations of copepods at r...,s2_r,1,-1,1,1,[genetic_change],unclear,[seafloor massive sulfide mining],plausibly_implied
287,WOS:000440879500047,Impacts of silicon addition on arsenic fractio...,s2_r,1,-1,1,1,[genetic_change],unclear,[pollution],plausibly_implied
317,WOS:000467513300003,Light penetration in a temperate meso-tidal la...,s2_r,1,-1,1,1,[species_change],unclear,[dredging],plausibly_implied
328,WOS:001445437800001,Metagenomic Analysis of the Microbial Communit...,s2_r,1,-1,1,1,[community_change],unclear,[heavy metal contamination],direct_primary_evidence
346,WOS:000348263100011,Nitrogen/carbon metabolism in response to NH4+...,s2_r,1,-1,1,1,"[species_change, community_change]",unclear,[pollution from sewage and agricultural runoff],direct_primary_evidence


## 6. Revised metrics if `-1` is excluded from eligible predictions

This is a post-processing what-if: keep the same model outputs, but only predict `ELIGIBLE` when all
four stage results are exactly `1`. Any `0`, `-1`, missing value, or parse failure becomes
`NOT_ELIGIBLE`.


In [50]:
def binary_screening_metrics(frame, pred_col, true_col="true_label"):
    pred_pos = frame[pred_col] == "ELIGIBLE"
    true_pos = frame[true_col] == "ELIGIBLE"

    tp = int((pred_pos & true_pos).sum())
    fp = int((pred_pos & ~true_pos).sum())
    fn = int((~pred_pos & true_pos).sum())
    tn = int((~pred_pos & ~true_pos).sum())

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "positive_support": tp + fn,
        "scored_rows": len(frame),
    }

scored_revised = scored.copy()
scored_revised["pred_label_no_minus_one"] = scored_revised[stage_cols].eq(1).all(axis=1).map(
    {True: "ELIGIBLE", False: "NOT_ELIGIBLE"}
)

metric_comparison = pd.DataFrame(
    [
        {"rule": "current: only 0 excludes", **binary_screening_metrics(scored_revised, "pred_label")},
        {"rule": "strict: 0 or -1 excludes", **binary_screening_metrics(scored_revised, "pred_label_no_minus_one")},
    ]
)

metric_comparison["precision_delta_vs_current"] = metric_comparison["precision"] - metric_comparison.loc[0, "precision"]
metric_comparison["recall_delta_vs_current"] = metric_comparison["recall"] - metric_comparison.loc[0, "recall"]
metric_comparison["f1_delta_vs_current"] = metric_comparison["f1"] - metric_comparison.loc[0, "f1"]
metric_comparison


,rule,tp,fp,fn,tn,precision,recall,f1,positive_support,scored_rows,precision_delta_vs_current,recall_delta_vs_current,f1_delta_vs_current
0,current: only 0 excludes,109,60,12,219,0.644970,0.900826,0.751724,121,400,0.000000,0.000000,0.000000
1,strict: 0 or -1 excludes,97,43,24,236,0.692857,0.801653,0.743295,121,400,0.047887,-0.099174,-0.008429


In [51]:
current_eligible_with_minus_one = scored_revised[
    (scored_revised["pred_label"] == "ELIGIBLE")
    & scored_revised[stage_cols].eq(-1).any(axis=1)
].copy()

current_eligible_with_minus_one["outcome_under_current_rule"] = current_eligible_with_minus_one["true_label"].map(
    {"ELIGIBLE": "TP", "NOT_ELIGIBLE": "FP"}
)

current_eligible_with_minus_one_summary = (
    current_eligible_with_minus_one["outcome_under_current_rule"]
    .value_counts()
    .rename_axis("current_outcome")
    .reset_index(name="count")
)
current_eligible_with_minus_one_summary


,current_outcome,count
0,FP,17
1,TP,12


In [52]:
strict_confusion = pd.crosstab(
    scored_revised["pred_label_no_minus_one"],
    scored_revised["true_label"],
    rownames=["strict prediction"],
    colnames=["truth"],
    dropna=False,
)
strict_confusion


truth,ELIGIBLE,NOT_ELIGIBLE
strict prediction,,
ELIGIBLE,97,43
NOT_ELIGIBLE,24,236


## 7. Exclusion-reason audit for false positives

The truth file includes an `exclusion_reason` for human-excluded records. This is more informative
than keyword trends because it explains why each false positive should have been excluded. The tables
below compare all current false positives with the remaining false positives after excluding any
`-1` stage prediction.


In [53]:
# Bring human exclusion reasons onto the scored dataframe.
truth_reasons = truth[["custom_id", "exclusion_reason"]].copy()
truth_reasons["exclusion_reason"] = truth_reasons["exclusion_reason"].fillna("").astype(str).str.strip()

scored_with_reasons = scored_revised.merge(
    truth_reasons,
    on="custom_id",
    how="left",
    validate="one_to_one",
)
scored_with_reasons["exclusion_reason"] = scored_with_reasons["exclusion_reason"].fillna("")

current_false_positives = scored_with_reasons[
    (scored_with_reasons["pred_label"] == "ELIGIBLE")
    & (scored_with_reasons["true_label"] == "NOT_ELIGIBLE")
].copy()

minus_one_false_positives = current_false_positives[
    current_false_positives[stage_cols].eq(-1).any(axis=1)
].copy()

strict_remaining_false_positives = scored_with_reasons[
    (scored_with_reasons["pred_label_no_minus_one"] == "ELIGIBLE")
    & (scored_with_reasons["true_label"] == "NOT_ELIGIBLE")
].copy()

pd.DataFrame([
    {"slice": "current false positives", "count": len(current_false_positives)},
    {"slice": "current FPs with any -1 stage", "count": len(minus_one_false_positives)},
    {"slice": "strict-rule remaining FPs", "count": len(strict_remaining_false_positives)},
])


,slice,count
0,current false positives,60
1,current FPs with any -1 stage,17
2,strict-rule remaining FPs,43


In [54]:
def exclusion_reason_bucket(reason):
    text = str(reason).lower().strip()
    if not text:
        return "missing/blank"
    if any(term in text for term in [
        "not related to biodiv",
        "not related to biodiversity",
        "not related to biodversity",
        "does not related",
        "clearly not related",
        "clearly unrelated",
        "unrelated",
    ]):
        return "not biodiversity-related"
    if any(term in text for term in [
        "no biodiver",
        "no biodiversity change",
        "no biodiv change",
        "no biodviersity change",
        "no bio change",
        "no direct biodiversity change",
        "no specific biodiv",
        "not bio change",
        "not any biodiversity change",
    ]):
        return "no biodiversity change endpoint"
    if any(term in text for term in [
        "molecular biology",
        "physiology",
        "plant nutition",
        "plant nutrition",
        "reproductive physiology",
        "fish physiology",
        "metabolite",
        "genetic",
        "crop",
        "yield",
        "leaf orientation",
    ]):
        return "organism/crop/physiology not biodiversity"
    if any(term in text for term in [
        "waste water",
        "wastewater",
        "sludge",
        "composting",
        "engineering process",
        "material science",
        "remediation",
        "antifouling",
        "baterial performance",
        "bacterial performance",
        "biofilm",
        "water additives",
    ]):
        return "engineering/remediation/process study"
    if any(term in text for term in [
        "no clear link",
        "driver",
        "anthropog",
        "anthro source",
        "natural process",
        "tectonic",
        "storm surge",
        "percipit",
        "precipitation",
        "climate",
    ]):
        return "driver/link not anthropogenic or unclear"
    if any(term in text for term in [
        "economics",
        "education",
        "psychology",
        "linguistics",
        "survey",
        "volunteer",
        "sense of place",
        "media",
    ]):
        return "social/economic/human study"
    return "other specific exclusion"

for frame in [current_false_positives, minus_one_false_positives, strict_remaining_false_positives]:
    frame["exclusion_bucket"] = frame["exclusion_reason"].map(exclusion_reason_bucket)

bucket_comparison = pd.concat(
    [
        current_false_positives["exclusion_bucket"].value_counts().rename("current_fp"),
        minus_one_false_positives["exclusion_bucket"].value_counts().rename("minus_one_fp"),
        strict_remaining_false_positives["exclusion_bucket"].value_counts().rename("strict_remaining_fp"),
    ],
    axis=1,
).fillna(0).astype(int)

bucket_comparison["strict_remaining_share"] = (
    bucket_comparison["strict_remaining_fp"] / len(strict_remaining_false_positives)
).fillna(0)

bucket_comparison.sort_values("strict_remaining_fp", ascending=False)


,current_fp,minus_one_fp,strict_remaining_fp,strict_remaining_share
exclusion_bucket,,,,
no biodiversity change endpoint,17,4,13,0.302326
driver/link not anthropogenic or unclear,16,4,12,0.279070
other specific exclusion,14,6,8,0.186047
not biodiversity-related,10,3,7,0.162791
engineering/remediation/process study,3,0,3,0.069767


In [55]:
def top_exclusion_reasons(frame, label, n=25):
    return (
        frame["exclusion_reason"]
        .replace("", "<blank>")
        .value_counts()
        .head(n)
        .rename_axis("exclusion_reason")
        .reset_index(name="count")
        .assign(slice=label)
        [["slice", "exclusion_reason", "count"]]
    )

pd.concat(
    [
        top_exclusion_reasons(current_false_positives, "current_fp"),
        top_exclusion_reasons(minus_one_false_positives, "minus_one_fp"),
        top_exclusion_reasons(strict_remaining_false_positives, "strict_remaining_fp"),
    ],
    ignore_index=True,
)


,slice,exclusion_reason,count
0,current_fp,not related to biodiversity,6
1,current_fp,No biodiversity change reported,4
2,current_fp,no biodiversity change reported,2
3,current_fp,not related to biodiversity change,2
4,current_fp,No biodiveristy change present,2
...,...,...,...
61,strict_remaining_fp,Dicuss how recreational visits to mountain par...,1
62,strict_remaining_fp,Text discusses garbage waste as anthro source....,1
63,strict_remaining_fp,"Text focuses on polychaetes, new record of spe...",1
64,strict_remaining_fp,"The focus of text is Bioremediation, technique...",1


In [56]:
title_col = "Article Title" if "Article Title" in strict_remaining_false_positives.columns else "title"
detail_cols = [
    "custom_id",
    title_col,
    "exclusion_bucket",
    "exclusion_reason",
    *stage_cols,
    "s1_bio",
    "s2_dir",
    "s3_drivers",
    "s4_link",
]
detail_cols = [col for col in detail_cols if col in strict_remaining_false_positives.columns]

strict_remaining_false_positives[detail_cols].sort_values(["exclusion_bucket", title_col])


,custom_id,Article Title,exclusion_bucket,exclusion_reason,s1_r,s2_r,s3_r,s4_r,s1_bio,s2_dir,s3_drivers,s4_link
162,WOS:000538150400092,Assessment of mobility and environmental risks...,driver/link not anthropogenic or unclear,Text discusses garbage waste as anthro source....,1,1,1,1,"[species_change, ecosystem_change]",negative,[pollution],assessed
166,WOS:000671387400009,Bioeroding (boring) polychaete species (Anneli...,driver/link not anthropogenic or unclear,"Text focuses on polychaetes, new record of spe...",1,1,1,1,"[species_change, ecosystem_change]",negative,[Lessepsian invasion (invasive species)],direct_primary_evidence
189,WOS:000429085900002,COMPARISON BETWEEN TREE DIEBACK OF PENDICULATE...,driver/link not anthropogenic or unclear,tree dieback is investigated in text due to mu...,1,1,1,1,[species_change],negative,"[forest_management, stand_structure]",direct_primary_evidence
177,WOS:000353486600014,Chemical and microbiological changes of salted...,driver/link not anthropogenic or unclear,Changes in Caspian Kutum (a fish) are discusse...,1,1,1,1,[community_change],positive,[salting/brining (preservation)],direct_primary_evidence
184,WOS:001602239300001,Colchicine-Induced Tetraploid Kenaf (Hibiscus ...,driver/link not anthropogenic or unclear,Text is focused Polyploidization (Process of p...,1,1,1,1,[genetic_change],positive,[colchicine treatment for polyploidization],direct_primary_evidence
193,WOS:000884680400004,Consequences of the accessibility of the mount...,driver/link not anthropogenic or unclear,Dicuss how recreational visits to mountain par...,1,1,1,1,[species_change],negative,"[tourism development, tourist traffic, infrast...",plausibly_implied
196,WOS:000415376000009,Costs of environmental degradation An analysis...,driver/link not anthropogenic or unclear,Text focuses on economics of env degradation n...,1,1,1,1,"[species_change, ecosystem_change]",negative,"[industrial_waste, hazardous_waste, pollution]",plausibly_implied
247,WOS:000329535100004,Exploring CO2 Sources and Sinks Nexus through ...,driver/link not anthropogenic or unclear,The link of anthro activities and green house ...,1,1,1,1,[ecosystem_change],negative,"[urban expansion / land cover change, energy c...",direct_primary_evidence
264,WOS:000416849800005,Ground-dwelling arthropod response to fire and...,driver/link not anthropogenic or unclear,species composition changes with wildfire and ...,1,1,1,1,[community_change],negative,[clearcutting],direct_primary_evidence
275,WOS:000320634200007,How will aspen respond to mountain pine beetle...,driver/link not anthropogenic or unclear,text is missing link of anthropogenic driver a...,1,1,1,1,"[species_change, community_change, ecosystem_c...",mixed,"[domestic ungulate browsing, wild ungulate bro...",plausibly_implied
